# 02. Legacy: группы котлов, сценарии и сравнение моделей

Старые analogs, AAKR и MSET: прежние параметры и нормализация. По умолчанию включено быстрое обновление калибровки; прежний режим доступен для сравнения.
Основной котёл выбирать не нужно: проверяются все котлы выбранных групп.
Результаты — в таблицах и графиках этого ноутбука с раскрывающейся навигацией.

[Настройки](#settings) · [План](#plan) · [Запуск](#run) · [Сводка](#overview) · [Графики](#details) ·
[Пороги](#thresholds) · [Близость](#proximity) · [Общие часы](#paired) · [Диагностика](#diagnostics)

Выполненный пользовательский прогон с графиками — в `02_Обучение_результаты.ipynb`.
Локальные библиотеки находятся в `lib/`; ничего устанавливать автоматически не требуется.


In [ ]:
from pathlib import Path
import os, sys, importlib
_notebook_parent = Path(globals().get('__file__', '02_Обучение.ipynb')).resolve().parent
_candidates = [_notebook_parent, Path.cwd() / 'legacy', Path.cwd().parent / 'legacy']
PROJECT_DIR = next((p for p in _candidates if (p / 'lib' / 'legacy_fleet.py').is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Откройте ноутбук из папки legacy с папкой lib.')
PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
for _p in (PROJECT_DIR, PROJECT_DIR / 'lib'):
    if str(_p) in sys.path: sys.path.remove(str(_p))
    sys.path.insert(0, str(_p))
__file__ = str(PROJECT_DIR / '02_Обучение.ipynb')
import legacy_group, legacy_metrics, legacy_fleet as LF
import legacy_plots
importlib.reload(legacy_plots)
importlib.reload(legacy_group)
importlib.reload(legacy_metrics)
LF = importlib.reload(LF)
from IPython.display import display, HTML
import pandas as pd
print(LF.VERSION, '·', PROJECT_DIR)


<a id="settings"></a>
## 0. Настройки

`local` — каждый котёл учится на своей истории; `pooled` — одна модель на истории группы;
`transfer` — обучение на одном котле, проверка на остальных котлах той же группы.
`TRANSFER_DONORS=None` перебирает всех доноров. Группу из одного котла можно включать в общий запуск.

`LOCAL_SENSOR_SETS=['common', 'full']`: индивидуальная модель на общем наборе для честного сравнения
и отдельно на собственном наборе датчиков. Общая модель и перенос всегда используют пересечение по группе,
включая небаланс пар/вода. Названия и роли берутся из Excel.


In [ ]:
CFG = LF.defaults()
CFG.update(
    GROUPS={'БКЗ-500': ['К-5', 'К-6', 'К-7']},
    # Пример трёх независимых групп:
    # GROUPS={'Группа 1': ['К-1','К-2','К-3','К-4'],
    #         'Группа 2': ['К-5','К-6','К-7'], 'Группа 3': ['К-8']},
    SELECT_GROUPS=None,                    # None — все группы; или ['Группа 2']
    SCENARIOS=['local', 'pooled', 'transfer'],
    LOCAL_SENSOR_SETS=['common', 'full'],
    TRANSFER_DONORS=None,                   # или {'БКЗ-500': ['К-5']}
    SENSOR_LIST_FILE='sensor_roles.xlsx',
    DATA_ROOT='opc_wide',
    DATA_PATHS={},                          # {'К-5': 'D:/data/К-5/hourly.parquet', ...}
    DATA_KIND='hourly', INSTANT_STEP='5min', INSTANT_PATHS={},
    FIRST_TEST='2025-03-01', FIRST_TEST_AUTO=False, EVAL_END=None,
    MAIN_METHODS=['analogs', 'AAKR', 'MSET'],
    OTHER_METHODS=[],
    # OTHER_METHODS=['AAKR (все датчики)', 'MSET (все датчики)', 'Регрессия',
    #                'Близнецы', 'Остатки analogs', 'Остатки AAKR', 'Остатки MSET'],
    RESIDUAL_BASES=['analogs', 'AAKR', 'MSET'],
    RESIDUAL_METHODS=['PCA', 'Махаланобис', 'Isolation Forest', 'AE', 'LSTM'],
    RESIDUAL_BAGGING=True, USE_NEURAL=True,
    RESIDUAL_AE_EPOCHS=30, RESIDUAL_LSTM_EPOCHS=10,
    RESIDUAL_LSTM_STEPS=24, RESIDUAL_MAX_TRAIN=8000,
    RUN_THRESHOLDS={},                      # необязательные пороги работы по котлам
    TRAIN_MODE='hours', TRAIN_HOURS=6500, TRAIN_MAX_DAYS=1095, TRAIN_DAYS=365,
    AAKR_H=0.5, MSET_H=0.7, MIN_TRAIN_COVER=0.5,
    CLEAN_WINDOW_H=168, CLEAN_MIN_HISTORY_H=24,
    CLEAN_SPIKE_FACTOR=8.0, CLEAN_FROZEN_H=24,  # фильтры используют только прошлые и текущие значения
    ALARM_Q=0.995, SMOOTH_H=6, PERSIST_H=3, MERGE_GAP_H=12,
    PRE_WINDOW_D=7, NEUTRAL_D=7, MIN_LEAD_H=3,
    WINDOWS_D=[1, 3, 7, 14, 30], ALARM_MATCH='overlap',
    THRESHOLD_MULTIPLIERS=[0.5, 0.75, 1.0, 1.25, 1.5, 2.0],
    THRESHOLD_VALIDATION_END=None,           # или дата раздела прошлой проверки и последующего контроля
    PROXIMITY_HALF_LIVES_D=[1, 3, 7, 14],
    RECOMPUTE=False, REUSE_PREVIOUS_MODEL=True, TRAIN_UPDATE='fast',
    FAST_MIN_CAL_H=72,                      # минимум часов ошибок для калибровки; иначе новый OOF-старт
    IMPORT_LEGACY_CACHE=False,              # кэш прежней очистки 2.0/2.1 несовместим с 2.2
    ALIGN_LOAD_ROLE=True,                   # сопоставить разные каналы расхода пара с ролью «нагрузка»
    RESUME_EXPERIMENTS=True, SHOW_PROGRESS_RESULTS=False,
    VERBOSE=False,                         # True — подробный журнал по месяцам
    HISTORY_MISSING='previous',             # previous / skip: экспорт не обучает пропущенные месяцы
    EXPORT=True, HISTORY_FOR=['analogs', 'AAKR', 'MSET'],
    LEGACY_OUTPUT_DIR='anomaly_results_legacy', LEGACY_MODELS_DIR='models_legacy',
    CONTINUE_ON_ERROR=True,
)
# JSON при ночном запуске переопределяет эти настройки.
CFG = LF.configuration(CFG)


<a id="plan"></a>
## 1. План и проверка входов

Таблица показывает каждый запуск: группа, сценарий, котёл проверки и котлы обучения.
Неприменимые сценарии одиночных групп не считаются ошибками.


In [ ]:
PLAN = LF.plan(CFG)
display(PLAN.drop(columns=['key']))
display(LF.preflight(CFG))
print('Экспериментов:', PLAN.status.eq('запланировано').sum())


<a id="run"></a>
## 2. Обучение и применение

Модель каждого месяца использует историю до начала этого месяца. Отсутствие данных одного донора
не запрещает использовать историю остальных. Модель и её калибровка общие для всех проверяемых котлов
при одинаковом составе доноров и датчиков; экспорт читает готовые модели.

**`TRAIN_UPDATE='fast'` — основной режим:**

1. Первый доступный месяц: прежние три OOF-блока для ошибок на незнакомых часах и итоговая модель (4 обучения).
2. Следующий месяц: предыдущая модель рассчитывает ошибки новых подходящих часов истории.
   Эти ошибки дополняют калибровку; ошибки часов, вышедших из обучающего окна или исключённых очисткой, удаляются.
   Затем одна модель обучается на текущем окне (1 обучение). При неизменном наборе часов — 0 обучений.
3. При изменении набора доступных датчиков или недостатке калибровочных ошибок выполняется новый OOF-старт.

Это ускоренное **обновление калибровки**, а не эквивалентный `partial_fit`. Масштабирование и память
основной модели пересчитываются на том же окне прежними методами. Порог и метрики могут отличаться:
ошибки теперь получены от предшествующей модели, а не от новых OOF-моделей каждого месяца.
`TRAIN_UPDATE='refit'` сохраняет прежние три OOF-блока каждый месяц. Для чистого сравнения задайте
`IMPORT_LEGACY_CACHE=False` в обоих запусках; папки `__fast` и `__refit` разделены.

Ускорены основные analogs/AAKR/MSET, их варианты «все датчики», регрессия и близнецы.
Для методов на остатках первая ступень тоже использует быстрый режим, но вторые ступени,
включая AE/LSTM, обучаются прежним способом: их нельзя просто соединить с изменённой шкалой остатков.
Никакого ограничения «1% времени тревог» здесь нет.

Очистка датчиков изменена: статистика скачков строится по предыдущим `CLEAN_WINDOW_H` часам,
без текущего значения, после накопления `CLEAN_MIN_HISTORY_H` часов. Порог скачка —
`CLEAN_SPIKE_FACTOR` масштабов IQR/1.349 приращений. Следующее значение не используется.
Залипание отмечается только после `CLEAN_FROZEN_H` часов; начало участка задним числом не маскируется.
Параметры очистки и её код входят в ключ кэша. Экспортированные модели содержат те же фильтры.
Это исправляет очистку датчиков; историческое определение режимов работы остаётся прежним.

**Модели и оценки 2.0/2.1 с прежней очисткой не импортируются**, даже если в старом CFG
осталось `IMPORT_LEGACY_CACHE=True`. Новый расчёт сохраняется в отдельных папках `__v22__`;
прежние результаты и экспорты остаются на месте.

Если основную модель обновить невозможно, используется предыдущая успешная, с датой её обучения.
Будущая или финальная модель для прошлого не используется. Экспорт пропущенного месяца берёт предыдущую
модель (`HISTORY_MISSING='previous'`) либо пропускает его (`'skip'`), не запускает обучение.
По умолчанию журнал содержит название метода и его итоги. Таблицы и графики выводятся в разделах ниже.
`VERBOSE=True` включает подробности по месяцам; `SHOW_PROGRESS_RESULTS=True` — промежуточные таблицы и графики. После прерывания повторный запуск
читает готовые отчёты и продолжает остальные эксперименты (`RESUME_EXPERIMENTS=True`).

В fast при пропуске проверяемого месяца перед следующим обновлением готовятся промежуточные месячные модели.
Так цепочка калибровки не зависит от того, на каком котле группы есть данные для проверки.
Дата начала проверки входит в ключ fast-кэша. Экспорт сам не запускает эти обновления.


In [ ]:
RESULT = LF.run(CFG)


<a id="overview"></a>
## 3. Основные метрики

PR AUC сохраняет прежнее название и расчёт `average_precision_score`.
Рядом показан **PR AUC случайный уровень**: доля часов перед аварией среди часов,
использованных для PR AUC (перед аварией + фон). Например, 0.10 означает базовый уровень 10%.
Он рассчитан на тех же доступных часах конкретной модели и окна; нейтральные часы исключены.
Recall — доля остановов с тревогой; Precision — доля полезных эпизодов среди полезных и фоновых.
Нейтральная полоса перед окном, как в legacy, не считается фоном; её эпизоды показаны отдельно в деталях.
F1 объединяет событийный Recall и эпизодный Precision по прежней формуле legacy; это разные единицы учёта.

При `overlap` продолжающаяся тревога засчитывается, даже если началась раньше окна.
При `start` — только начавшаяся в окне. Не вводятся ограничения по минимальной полноте окна перед остановом.
Процент фона в тревоге считается по подтверждённым эпизодам, а не по одиночным превышениям порога.
Упреждение — от начала эпизода, как в legacy; оно не означает установленную дату появления дефекта.

Рядом выведены количества полезных, фоновых и нейтральных эпизодов, часы фона и реально оценённые рабочие часы.
«% фона в тревоге» показан в процентах (например, 12.5 означает 12.5%); PR AUC, Precision и Recall — доли от 0 до 1.


In [ ]:
LF.show_overview(RESULT)


<a id="details"></a>
## 4. Графики и каждый останов

Раскрывайте группу, сценарий, котёл и метод. Возвращён график из прежнего ноутбука:
сверху — **суточный максимум ошибки / порога**, серые простои и цветные периоды
аварийного ремонта, планового ремонта и резерва. В заголовке — PR AUC, случайный
уровень, пойманные аварии, частота фоновых тревог и доля фона в тревоге.
Снизу — прежняя общая полоса «котёл / тревога»: зелёная работа, серый простой,
красные эпизоды тревог. Цвета, подписи и легенда прежние.

Сохранено только новое масштабирование оценки: до порога 1 шкала линейная,
выше 1 — сжатая. Пики не обрезаются на 5. Дополнительных цветных окон,
третьей панели и раздельной окраски тревог нет.

Ниже можно выбрать даты без обучения. Обрезка графика не пересчитывает тревоги;
метрики в заголовке относятся ко всему периоду эксперимента.


In [ ]:
# Загрузить заменённый файл графиков с диска, даже если ядро уже работало.
import importlib
import legacy_plots
importlib.invalidate_caches()
legacy_plots = importlib.reload(legacy_plots)
print('Отрисовка:', legacy_plots.PLOT_VERSION, '·', legacy_plots.__file__)
LF.show_details(RESULT)


### Отдельный период
Выберите эксперимент и даты. Для первого просмотра настройки берутся из первого готового эксперимента; затем их можно менять.
Раскомментируйте последнюю строку, чтобы вывести график выбранного периода.


In [ ]:
if RESULT['records']:
    _record = RESULT['records'][0]
    DETAIL_PERIOD = dict(
        group=_record['meta']['group'], target=_record['meta']['target'],
        scenario=_record['meta']['scenario'], donors=_record['meta']['donors'],
        view=next(iter(_record['views'])),
        start=None, end=None,  # например, start='2025-06-01', end='2025-07-01' (правая граница не включается)
    )
    # LF.show_period(RESULT, **DETAIL_PERIOD)


<a id="thresholds"></a>
## 5. Сравнение порогов — без обучения заново

1.0 — исходный порог каждого месяца; 0.75 — на 25% ниже, 1.5 — на 50% выше.
Для каждого варианта заново формируются тревоги и считаются Precision, Recall, F1, число и длительность фоновых тревог.
PR AUC при умножении всех порогов на одну константу остаётся тем же.

Это диагностическая таблица, автоматического выбора самого красивого порога по будущим остановам нет.
Если задан `THRESHOLD_VALIDATION_END`, отдельно показываются прошлая проверка и последующий контроль.
Выбирать порог следует по прошлой части, оценивать выбранный вариант — по последующей.


In [ ]:
LF.show_table(RESULT, 'thresholds')


<a id="proximity"></a>
## 6. Близость к следующему останову — отдельный эксперимент

Вес часа = `2 ** (-дни_до_останова / полураспад_веса)` перед следующим аварийным остановом;
перед следующей плановой заявкой/резервом вес равен нулю. За полураспад вес уменьшается вдвое.
PR близости строится по плавным весам вместо двоичной разметки. Это **не обычный PR AUC** и не вероятность аварии.
Все одинаковые оценки включаются одновременно. Постоянный сигнал даёт средний вес, показанный в столбце «базовый вес».
Часы после последней известной заявки не оцениваются: неизвестно, когда следующий останов.
Последние `MIN_LEAD_H` часов исключаются. Метрика оценивает связь сигнала с приближением останова;
сама по себе она не поощряет наиболее полезное раннее предупреждение.


In [ ]:
LF.show_table(RESULT, 'proximity')


## 7. Зависимость от окна предупреждения

Для каждого окна отдельно показаны те же основные метрики и фактическое число часов фона.
Сравнивать сценарии нужно внутри одного окна. Увеличение окна не является улучшением самой модели.


In [ ]:
LF.show_table(RESULT, 'windows')


<a id="paired"></a>
## 8. Индивидуальная и общая модель на одинаковых часах

Каждая пара сравнивается отдельно на общем наборе датчиков и пересечении реально рассчитанных часов.
Общее пересечение сразу всех сценариев не используется. Эпизоды тревог формируются на полном ряду,
поэтому исключение часов из сравнения не создаёт новые начала тревог.


In [ ]:
PAIRED = LF.paired_comparison(RESULT)
display(HTML(LF._table(PAIRED)))


<a id="diagnostics"></a>
## 9. Диагностика и сохранённые модели

Причины отсутствия результата показаны отдельно от основных метрик. Пропуски не объявляются нормой.
В разделе графиков есть таблица происхождения моделей: обучение, кэш или предыдущая модель.
Финальные модели — `models_legacy/<группа>/<эксперимент>/<котёл>/<метод>/final/`,
исторические — соседние папки `ГГГГ-ММ`. Готовые модели можно открыть в ноутбуке 03.
Служебный `fleet_report.joblib` сохраняет оценки и данные отчёта, чтобы восстановить таблицы без обучения.

Для просмотра уже готовых экспериментов после прерывания без продолжения обучения: `RESULT = LF.load_report(CFG)`.


In [ ]:
LF.show_table(RESULT, 'hours')
if RESULT['errors']:
    display(pd.DataFrame(RESULT['errors'])[['group','scenario','target','donors','error']])
print('Ошибок:', len(RESULT['errors']))
print('Моделей экспортировано:', sum(len(r['exports']) for r in RESULT['records']))
# Для восстановления после перезапуска ядра вместо обучения:
# RESULT = LF.load_report(CFG)
# Для собственного анализа доступны RESULT['tables']['main'/'thresholds'/'proximity'/'events'/'windows'/'hours'].
